# 04 — Evaluation, Model Comparison and Example Demo
Combine actual metric JSON files, identify the best macro-F1 model, review errors, and run a sample prediction.

In [ ]:
from pathlib import Path
import sys, os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/"src").exists() and (ROOT.parent/"src").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
DATA_DIR = ROOT/"data"/"raw"
OUTPUT_DIR = ROOT/"outputs"
for d in ["plots","confusion_matrices","metrics","predictions"]: (OUTPUT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Project root:", ROOT, "| Dataset:", DATA_DIR)

In [ ]:
from src.evaluation import compare_metrics
metric_files=[OUTPUT_DIR/"metrics"/f"{name}_metrics.json" for name in ["svm","rf","mobilenetv2"]]
metric_files=[path for path in metric_files if path.exists()]
print("Metric files:",[p.name for p in metric_files])
if not metric_files:
    raise FileNotFoundError("No model metrics found. Run the training notebooks first.")
comparison=compare_metrics(metric_files,OUTPUT_DIR/"metrics"/"model_comparison.csv")
display(comparison)
eligible=comparison[comparison.validation_macro_f1.notna()]
if not eligible.empty:
    best=eligible.sort_values(["validation_macro_f1","model"],ascending=[False,True]).iloc[0]
    print(f"Selected by validation macro F1: {best.get('model')} | validation={best.get('validation_macro_f1'):.4f} | held-out test={best.get('f1_macro'):.4f}")

## Example run
Select a held-out test image. The code below calls the same prediction functions as `main.py`. If model files are missing, train the relevant approach first.

In [ ]:
from src.dataset import discover_dataset, stratified_split
from src.deep_learning_model import load_mobilenet
from src.feature_extraction import extract_features
import joblib, torch
from PIL import Image
from torchvision import transforms
test_image = test_path = None
if "test_df" not in globals():
    _,_,test_df = stratified_split(discover_dataset(DATA_DIR))
test_image = test_df.iloc[0].path
print("Demo image:",test_image,"actual label:",test_df.iloc[0].label)
for name in ["svm","rf"]:
    path=ROOT/"models"/f"{name}_model.joblib"
    if path.exists():
        bundle=joblib.load(path); x=extract_features(test_image).reshape(1,-1)
        pred=bundle["pipeline"].predict(x)[0]
        proba=bundle["pipeline"].predict_proba(x)[0] if hasattr(bundle["pipeline"],"predict_proba") else None
        print(name.upper(),pred,"confidence:",float(proba.max()) if proba is not None else "not available")
dl_path=ROOT/"models"/"mobilenetv2.pth"
if dl_path.exists():
    net,classes,device=load_mobilenet(dl_path)
    tf=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),
        transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    x=tf(Image.open(test_image).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad(): probs=torch.softmax(net(x),dim=1)[0]
    print("MobileNetV2",classes[int(probs.argmax())],"probability",float(probs.max()))
plt.figure(figsize=(5,5)); plt.imshow(Image.open(test_image).convert("RGB")); plt.title(Path(test_image).name); plt.axis("off"); plt.show()

## Interpretation
Select the candidate using validation macro F1 and validation-loss checkpointing. Report test metrics only after that choice is fixed; do not select a model from test scores. Review class-wise recall, confusion matrices, and ROC curves. Report actual inference latency measured on documented hardware; do not reuse generic benchmark figures as if measured locally.